# queue

> Selection queue route handlers for Phase 1

In [ ]:
#| default_exp routes.selection.queue

In [ ]:
#| export
from cjm_fasthtml_interactions.core.state_store import get_session_id

from cjm_fasthtml_workflow_transcript_decomp.routes.models import SelectionUrls
from cjm_fasthtml_workflow_transcript_decomp.routes.selection.core import (
    _get_step_state, _update_step_state, _build_queue_response
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_selection.preview_panel import (
    _render_preview_panel
)
from cjm_fasthtml_workflow_transcript_decomp.services.source_utils import (
    select_all_in_group, reorder_sources
)

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

## Add/Remove Handlers

In [ ]:
#| export
def _handle_selection_add(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    job_id: str,  # Job ID to add
    plugin_name: str,  # Plugin name for the source
    urls: SelectionUrls,  # URL bundle for rendering
):  # Queue component with OOB stats, optionally with OOB source list
    """Add a source to the selection queue."""
    session_id = get_session_id(sess)
    step_state = _get_step_state(workflow, session_id)
    selected_sources = step_state.get("selected_sources", [])
    
    # Check if already selected
    if not any(s.get("job_id") == job_id for s in selected_sources):
        selected_sources.append({"job_id": job_id, "plugin_name": plugin_name})
        _update_step_state(workflow, session_id, selected_sources)
    
    return _build_queue_response(workflow, session_id, selected_sources, urls)

In [ ]:
#| export
def _handle_selection_remove(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    job_id: str,  # Job ID to remove
    urls: SelectionUrls,  # URL bundle for rendering
):  # Queue component with OOB stats, optionally with OOB source list
    """Remove a source from the selection queue."""
    session_id = get_session_id(sess)
    step_state = _get_step_state(workflow, session_id)
    selected_sources = step_state.get("selected_sources", [])
    
    # Remove the item
    selected_sources = [s for s in selected_sources if s.get("job_id") != job_id]
    _update_step_state(workflow, session_id, selected_sources)
    
    return _build_queue_response(workflow, session_id, selected_sources, urls)

## Reorder/Clear Handlers

In [ ]:
#| export
async def _handle_selection_reorder(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls: SelectionUrls,  # URL bundle for rendering
):  # Updated queue component
    """Reorder items in the selection queue based on SortableJS result."""
    session_id = get_session_id(sess)
    step_state = _get_step_state(workflow, session_id)
    selected_sources = step_state.get("selected_sources", [])
    
    # Extract new order from form data (hidden inputs sent in DOM order after drag-drop)
    form_data = await request.form()
    new_order_ids = form_data.getlist("item")
    
    selected_sources = reorder_sources(selected_sources, new_order_ids)
    _update_step_state(workflow, session_id, selected_sources)
    
    return _build_queue_response(
        workflow, session_id, selected_sources, urls,
        include_stats=False, include_source_list=False,
    )

In [ ]:
#| export
def _handle_selection_clear(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls: SelectionUrls,  # URL bundle for rendering
):  # Queue component with OOB stats, optionally with OOB source list
    """Clear all items from the selection queue."""
    session_id = get_session_id(sess)
    _update_step_state(workflow, session_id, [])
    
    return _build_queue_response(workflow, session_id, [], urls)

## Select All Handler

In [ ]:
#| export
def _handle_selection_select_all(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    group_key: str,  # Group key to select all transcriptions for
    grouping_mode: str,  # Current grouping mode: "audio_path" or "batch_id"
    urls: SelectionUrls,  # URL bundle for rendering
):  # Queue component with OOB stats, optionally with OOB source list
    """Select all transcriptions for a given group."""
    session_id = get_session_id(sess)
    step_state = _get_step_state(workflow, session_id)
    selected_sources = step_state.get("selected_sources", [])
    
    all_transcriptions = workflow.source_service.query_transcriptions(limit=500)
    selected_sources = select_all_in_group(all_transcriptions, group_key, grouping_mode, selected_sources)
    
    _update_step_state(workflow, session_id, selected_sources=selected_sources)
    
    return _build_queue_response(
        workflow, session_id, selected_sources, urls, grouping_mode=grouping_mode,
    )

## Preview Handler

In [ ]:
#| export
def _handle_selection_preview(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    job_id: str,  # Job ID to preview
    plugin_name: str,  # Plugin name for the source
):  # Full preview panel component (collapsible, open with content)
    """Get preview panel for a selected source."""
    # Get the transcription from the source service
    source_block = workflow.source_service.get_transcription_by_id(job_id, plugin_name)
    
    if not source_block:
        # Return preview panel with error state
        return _render_preview_panel(
            preview_job_id=job_id,
            preview_text="Could not load preview content.",
            is_open=True
        )
    
    # Return the full preview panel with content and open state
    return _render_preview_panel(
        preview_job_id=job_id,
        preview_text=source_block.text,
        is_open=True
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()